<a href="https://colab.research.google.com/github/aansikkaw/ecommerce-demand-forecasting/blob/main/E_Commerce_Demand_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Load and Clean Data
df = pd.read_csv('Online Retail.csv')
df.head()





In [ ]:
df

In [ ]:
df.shape
df.columns

In [ ]:
df.isnull().sum()


In [ ]:
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

In [ ]:
df.info()
df.describe()

In [ ]:
# 1. Filter the dataset to show only rows where Quantity is less than zero
negative_quantities = df[df['Quantity'] < 0]

# 2. Sort the values from lowest (most negative) to highest to easily spot the massive outliers
negative_quantities = negative_quantities.sort_values(by='Quantity')

# 3. Display the top 20 largest returns/cancellations
print("Top 20 Largest Negative Quantities (Returns/Cancellations):")
print(negative_quantities[['InvoiceNo', 'CustomerID', 'Description', 'Quantity', 'UnitPrice']].head(20))

In [ ]:

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'].astype(str), errors='coerce')
df = df.dropna(subset=['InvoiceDate'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['TotalSales'] = df['Quantity'] * df['UnitPrice']
df['Month'] = df['InvoiceDate'].dt.to_period('M')
df['Month_Number'] = df['InvoiceDate'].dt.month
df

In [ ]:
monthly_sales = df.groupby('Month')['TotalSales'].sum()
top_products = df.groupby('Description')['Quantity'].sum().nlargest(10)
country_sales = df.groupby('Country')['TotalSales'].sum().nlargest(5)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Feature Engineering for EDA (Calculating Revenue)
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['Month_Year'] = df['InvoiceDate'].dt.to_period('M')
df['Month'] = df['InvoiceDate'].dt.month

# Set global visual style for the portfolio
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('E-Commerce Exploratory Data Analysis (EDA)', fontsize=20, fontweight='bold')

# 2. Sales Trend Over Time (Top Left)
sales_trend = df.groupby('Month_Year')['Revenue'].sum().reset_index()
sales_trend['Month_Year'] = sales_trend['Month_Year'].astype(str)
sns.lineplot(ax=axes[0, 0], data=sales_trend, x='Month_Year', y='Revenue', marker='o', color='b', linewidth=2.5)
axes[0, 0].set_title('1. Macro Sales Trend Over Time', fontsize=14)
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].set_ylabel('Total Revenue ($)')

# 3. Top-Selling Products (Top Right)
top_products = df.groupby('Description')['Quantity'].sum().nlargest(10).reset_index()
sns.barplot(ax=axes[0, 1], data=top_products, y='Description', x='Quantity', palette='viridis')
axes[0, 1].set_title('2. Top 10 Best-Selling Products (Volume)', fontsize=14)
axes[0, 1].set_xlabel('Total Units Sold')
axes[0, 1].set_ylabel('')

# 4. Region-Wise Performance (Bottom Left)
# Excluding UK to see international spread, or keep all. We'll show top 10 regions.
top_regions = df.groupby('Country')['Revenue'].sum().nlargest(10).reset_index()
sns.barplot(ax=axes[1, 0], data=top_regions, x='Country', y='Revenue', palette='magma')
axes[1, 0].set_title('3. Region-Wise Performance (Top 10 Markets)', fontsize=14)
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].set_ylabel('Total Revenue ($)')

# 5. Seasonal Demand Patterns (Bottom Right)
seasonal_trend = df.groupby('Month')['Quantity'].sum().reset_index()
sns.barplot(ax=axes[1, 1], data=seasonal_trend, x='Month', y='Quantity', color='teal')
axes[1, 1].set_title('4. Seasonal Demand Patterns (Aggregate Monthly Volume)', fontsize=14)
axes[1, 1].set_xlabel('Month of the Year (1 = Jan, 12 = Dec)')
axes[1, 1].set_ylabel('Total Units Sold')

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

In [ ]:
# Feature Engineering for ML
# Aggregate data at the Product-Month level
demand_df = df.groupby(['Description', 'Month']).agg({'Quantity': 'sum'}).reset_index()
demand_df['Month'] = demand_df['Month'].astype(str)
demand_df.head()



In [ ]:
# Create lag features (previous month's demand) to predict current demand
demand_df['Lag_1_Quantity'] = demand_df.groupby('Description')['Quantity'].shift(1)
demand_df.dropna(inplace=True)
demand_df.head()

In [ ]:


X = demand_df[['Lag_1_Quantity']]
y = demand_df['Quantity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model 1: Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

# Model 2: Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

# Evaluation
print(f"Linear Regression MAE: {mean_absolute_error(y_test, lr_preds)}")
print(f"Random Forest MAE: {mean_absolute_error(y_test, rf_preds)}")

In [ ]:
from xgboost import XGBRegressor
demand_df = df.groupby(['Description', pd.Grouper(key='InvoiceDate', freq='ME')])['Quantity'].sum().reset_index()
demand_df['Month'] = demand_df['InvoiceDate'].dt.month

demand_df['Lag_1'] = demand_df.groupby('Description')['Quantity'].shift(1)
demand_df['Lag_2'] = demand_df.groupby('Description')['Quantity'].shift(2)
demand_df['Lag_3'] = demand_df.groupby('Description')['Quantity'].shift(3)
demand_df['Rolling_Mean_3'] = demand_df.groupby('Description')['Lag_1'].transform(lambda x: x.rolling(3, min_periods=1).mean())

demand_df = demand_df.dropna()

upper_limit = demand_df['Quantity'].quantile(0.99)
demand_df = demand_df[demand_df['Quantity'] <= upper_limit]

X = demand_df[['Lag_1', 'Lag_2', 'Lag_3', 'Rolling_Mean_3', 'Month']]
y = demand_df['Quantity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb = XGBRegressor(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42)
xgb.fit(X_train, y_train)

xgb_preds = xgb.predict(X_test)
mae = mean_absolute_error(y_test, xgb_preds)

print(f"XGBoost MAE: {mae}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Feature Importance Analytics for XGBoost (updated from Random Forest)
# Reconstruct the feature list based on your input matrix for XGBoost
feature_names = ['Lag_1', 'Lag_2', 'Lag_3', 'Rolling_Mean_3', 'Month']

xgb_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': xgb.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x='Importance', y='Feature', hue='Feature', data=xgb_importance_df, palette='viridis', legend=False)
plt.title('Feature Importance Breakdown (XGBoost Demand Model)')
plt.xlabel('Gain (Feature Importance)')
plt.ylabel('Feature Matrix')
plt.show()

# 2. Residual Analysis (Prediction Error Distribution) for XGBoost (updated from Random Forest)
# Residuals = Actual Demand - Predicted Demand
xgb_residuals = y_test - xgb_preds

plt.figure(figsize=(10, 5))
sns.histplot(xgb_residuals, kde=True, color='teal', bins=50)
plt.axvline(x=0, color='black', linestyle='--', linewidth=2)
plt.title('XGBoost Residual Distribution Analysis')
plt.xlabel('Prediction Error (Actual - Predicted Units)')
plt.ylabel('Frequency Density')
plt.show()

In [ ]:
X_predict = demand_df[['Lag_1', 'Lag_2', 'Lag_3', 'Rolling_Mean_3', 'Month']]
demand_df['Predicted_Demand'] = xgb.predict(X_predict)
demand_df['Predicted_Demand'] = demand_df['Predicted_Demand'].round().astype(int)

# The 'Month' column in the current demand_df is an integer (1-12).
# So, latest_month should be the maximum month number.
latest_month_num = demand_df['Month'].max()
forecast_df = demand_df[demand_df['Month'] == latest_month_num].sort_values(by='Predicted_Demand', ascending=False)

print("\n========================================================")
print("             DEMAND FORECASTING OUTPUT")
print("========================================================")
print(forecast_df[['Description', 'Lag_1', 'Predicted_Demand']].head(15).to_string(index=False))

print("\n========================================================")
print("      BUSINESS INSIGHTS FOR INVENTORY OPTIMIZATION")
print("========================================================")
print("1. High-Priority Restocking:")
top_item_1 = forecast_df.iloc[0]['Description']
top_item_2 = forecast_df.iloc[1]['Description']
print(f"   - Immediately secure inventory for '{top_item_1}' and '{top_item_2}'.")
print("   - Allocate premium warehouse space for these fast-moving goods to reduce picking time.")
print("\n2. Seasonal Capacity Planning:")
print("   - Align logistics and warehouse labor with the peak transaction months identified in the seasonal EDA chart.")
print("   - Draft supplier contracts 6 weeks prior to anticipated seasonal surges to avoid stockouts.")
print("\n3. Capital Allocation:")
print("   - Halt or reduce procurement for items in the bottom 10% of the Predicted_Demand array.")
print("   - Liquidate dead stock identified by consistently zero or negative growth in Lag_1 to free up working capital.")